# BOHDI LoRA Smoke Test (Colab T4)

Runs the full pipeline (generate traces -> filter -> train LoRA -> eval) end-to-end
on 3 examples. Takes ~15-25 min on a free T4 (if it fits).

**Colab-specific tweaks vs `smoke.sh`:**
- Uses **fp16** (not bf16) for training — T4 (Turing) has no native bf16.
- Model: currently E4B (default). E4B weights are ~15GB on disk, so it'll likely OOM on a 15GB T4 — if it crashes, change `SMOKE_MODEL` in cell 0 to `google/gemma-3n-E2B-it` and re-run.

**Before running:** Runtime > Change runtime type > select **T4 GPU**.

In [ ]:
# 0. Verify GPU and pin model choice
# Trying E4B first. Weights are ~15GB in bf16, T4 has 15GB VRAM — no headroom
# for activations / KV cache, so OOM is very likely. If CUDA OOM hits, swap to
# "google/gemma-3n-E2B-it" (~5GB) and re-run from cell 0.
SMOKE_MODEL = "google/gemma-3n-E4B-it"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 1. Set your HF token and VERIFY it works.
# Hard-stops if HF_TOKEN is missing, placeholder, or unauthorized.
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    pass

if not token:
    token = "hf_PASTE_YOUR_TOKEN_HERE"  # <-- edit if not using Colab Secrets

if not token or token == "hf_PASTE_YOUR_TOKEN_HERE":
    raise SystemExit(
        "HF_TOKEN not set. Add it to Colab Secrets (key icon in sidebar) or paste above.\n"
        "Get a token: https://huggingface.co/settings/tokens\n"
        f"Accept terms: https://huggingface.co/{SMOKE_MODEL}"
    )
if not token.startswith("hf_"):
    raise SystemExit("HF_TOKEN doesn't look like an HF token (should start with 'hf_')")

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

!pip install -q huggingface_hub
from huggingface_hub import HfApi
api = HfApi(token=token)
try:
    who = api.whoami()
    print(f"Authenticated as: {who['name']}")
except Exception as e:
    raise SystemExit(f"HF auth failed: {e}")

try:
    api.model_info(SMOKE_MODEL)
    print(f"Access to {SMOKE_MODEL} confirmed")
except Exception as e:
    raise SystemExit(
        f"Cannot access {SMOKE_MODEL}: {e}\n"
        f"Accept terms at https://huggingface.co/{SMOKE_MODEL} while logged in as this account."
    )

In [ ]:
# 2. Clone repo and install deps
!git clone https://github.com/PeterLi-jpg/bohdi-lora.git
%cd bohdi-lora
!pip install -q -r requirements.txt

In [ ]:
# 3. Patch the smoke training config for Colab T4:
#    - inject SMOKE_MODEL from cell 0 (so config + CLI stay in sync)
#    - bf16 -> fp16 (T4 has no native bf16)
import yaml, shutil

src = "configs/lora_gemma_smoke.yaml"
dst = "configs/lora_gemma_smoke_colab.yaml"
shutil.copy(src, dst)

with open(dst) as f:
    cfg = yaml.safe_load(f)
cfg["model"]["name"] = SMOKE_MODEL
cfg["training"]["bf16"] = False
cfg["training"]["fp16"] = True
with open(dst, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Wrote {dst}:")
!cat {dst}

In [ ]:
# 4. Setup directories and download HealthBench data
!mkdir -p logs data/raw data/sft/smoke eval/smoke checkpoints
!python scripts/download_data.py

In [ ]:
# 5. Generate 3 BOHDI traces (model download is the slow part, ~3-5 min)
!python scripts/generate_traces.py \
    --model $SMOKE_MODEL \
    --datasets healthbench_hard \
    --output data/sft/smoke/raw_traces.jsonl \
    --use-bodhi \
    --max-examples 3

In [ ]:
# 6. Grade and filter traces (uses 0.5B ungated grader)
!python scripts/filter_traces.py \
    --input data/sft/smoke/raw_traces.jsonl \
    --healthbench-data data/raw/healthbench_hard.jsonl \
    --grader-model Qwen/Qwen2.5-0.5B-Instruct \
    --output-dir data/sft/smoke \
    --min-score -999 \
    --val-ratio 0.34

In [ ]:
# 7. Train LoRA (1 epoch on smoke set, using patched Colab config)
!python scripts/train_lora.py --config configs/lora_gemma_smoke_colab.yaml

In [ ]:
# 8. Eval with LoRA adapter
!python scripts/eval_healthbench.py \
    --model $SMOKE_MODEL \
    --lora-path checkpoints/best \
    --sample-ids data/raw/hard_200_sample_ids.json \
    --grader-model Qwen/Qwen2.5-0.5B-Instruct \
    --output eval/smoke/lora.json \
    --max-examples 3

In [ ]:
# 9. U-shape stratification
!python scripts/eval_ushape.py \
    --eval-jsons eval/smoke/lora.json \
    --healthbench data/raw/healthbench_hard.jsonl \
    --tertile-on-holdout-only \
    --output eval/smoke/ushape.json

In [ ]:
# 10. Check results
import json

print("=== Eval Results ===")
with open("eval/smoke/lora.json") as f:
    res = json.load(f)
    for k in ["mean", "std", "brier", "ece"]:
        if k in res:
            print(f"  {k}: {res[k]:.4f}")

print("\n=== U-Shape ===")
with open("eval/smoke/ushape.json") as f:
    print(json.dumps(json.load(f), indent=2)[:2000])

print("\nSmoke test PASSED!")